# Week 1 - EDA & Data Ingestion

In [4]:
pip install statsmodels

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf

# Loading dataset

In [6]:
df = pd.read_csv("restaurant_dataset.csv")
print(df)

FileNotFoundError: [Errno 2] No such file or directory: 'restaurant_dataset.csv'

# EDA

In [ ]:
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("\nSummary of dataset:")
df.info()

print("\nStatistics:")
df.describe()

print("\nShape:")
df.shape

In [ ]:
print("\nMissing values:")
df.isnull().sum()

# Date-Time conversion

In [ ]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Remove rows with invalid dates
df = df.dropna(subset=["date"]).copy()

In [ ]:
# Aggregate to daily sales
# Since this dataset has multiple rows per day, create a daily time series
daily_sales = (
    df.groupby("date", as_index=False)["quantity_sold"]
      .sum()
      .rename(columns={"quantity_sold": "daily_quantity_sold"})
)

# Sort by date
daily_sales = daily_sales.sort_values("date")

# Set datetime index
daily_sales = daily_sales.set_index("date")

print("\nDaily aggregated data:")
print(daily_sales.head())

In [ ]:
# Make datetime index continuous

full_range = pd.date_range(
    start=daily_sales.index.min(),
    end=daily_sales.index.max(),
    freq="D"
)

daily_sales = daily_sales.reindex(full_range)
daily_sales.index.name = "date"

print("\nMissing dates after reindexing:",
      daily_sales["daily_quantity_sold"].isna().sum())

# Fill missing dates
daily_sales["daily_quantity_sold"] = daily_sales["daily_quantity_sold"].fillna(0)

print("\nAfter filling missing dates:")
print(daily_sales.head())

# Overall sales trend

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(daily_sales.index, daily_sales["daily_quantity_sold"])
plt.title("Overall Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Daily Quantity Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#  Weekly and monthly patterns

weekly_sales = daily_sales["daily_quantity_sold"].resample("W").sum()
monthly_sales = daily_sales["daily_quantity_sold"].resample("M").sum()

plt.figure(figsize=(10, 5))
plt.plot(weekly_sales.index, weekly_sales.values)
plt.title("Weekly Sales Pattern")
plt.xlabel("Week")
plt.ylabel("Weekly Quantity Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(monthly_sales.index, monthly_sales.values)
plt.title("Monthly Sales Pattern")
plt.xlabel("Month")
plt.ylabel("Monthly Quantity Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition
# Weekly seasonality -> period=7 for daily data
decomposition = seasonal_decompose(
    daily_sales["daily_quantity_sold"],
    model="additive",
    period=7
)

fig = decomposition.plot()
fig.set_size_inches(10, 5)
plt.tight_layout()
plt.show()

In [ ]:
#  Autocorrelation analysis

plt.figure(figsize=(10, 5))
autocorrelation_plot(daily_sales["daily_quantity_sold"])
plt.title("Autocorrelation Plot")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plot_acf(daily_sales["daily_quantity_sold"], lags=30)
plt.title("ACF - Daily Sales")
plt.tight_layout()
plt.show()

# Week 2 – Feature Engineering & Data Preparation

This week we build the features the model will actually learn from:
1. **Rolling-window statistics** — capture recent trends
2. **Lag features** — give the model historical context
3. **Encoding categorical columns** — convert text labels to numbers
4. **Train / test split** — evaluate on unseen future data

## 1. Rolling-Window Statistics

Rolling statistics summarise the recent past over a sliding window.
They help the model understand **trends** and **local volatility** without hard-coding fixed look-back periods.

In [ ]:
# ── Rolling-window features ──────────────────────────────────────────────────
# Work on a fresh copy so the original daily_sales is untouched
features_df = daily_sales.copy()

# 7-day rolling mean & std  (captures weekly trend)
features_df["rolling_mean_7"]  = features_df["daily_quantity_sold"].rolling(window=7).mean()
features_df["rolling_std_7"]   = features_df["daily_quantity_sold"].rolling(window=7).std()

# 14-day rolling mean & std  (captures bi-weekly trend)
features_df["rolling_mean_14"] = features_df["daily_quantity_sold"].rolling(window=14).mean()
features_df["rolling_std_14"]  = features_df["daily_quantity_sold"].rolling(window=14).std()

# 30-day rolling mean  (captures monthly trend)
features_df["rolling_mean_30"] = features_df["daily_quantity_sold"].rolling(window=30).mean()

print("Rolling-window features added:")
print(features_df[["daily_quantity_sold",
                    "rolling_mean_7", "rolling_std_7",
                    "rolling_mean_14", "rolling_std_14",
                    "rolling_mean_30"]].head(35))


In [ ]:
# ── Visualise rolling means alongside raw sales ───────────────────────────────
plt.figure(figsize=(14, 5))
plt.plot(features_df.index, features_df["daily_quantity_sold"],
         alpha=0.4, label="Daily Sales", color="steelblue")
plt.plot(features_df.index, features_df["rolling_mean_7"],
         label="7-day Rolling Mean",  color="orange")
plt.plot(features_df.index, features_df["rolling_mean_14"],
         label="14-day Rolling Mean", color="green")
plt.plot(features_df.index, features_df["rolling_mean_30"],
         label="30-day Rolling Mean", color="red")
plt.title("Daily Sales with Rolling Means")
plt.xlabel("Date")
plt.ylabel("Quantity Sold")
plt.legend()
plt.tight_layout()
plt.show()


## 2. Lag Features

A **lag feature** is simply the value of a variable at a previous time step.
Lag-1 = yesterday's sales, Lag-7 = sales one week ago, etc.
They give the model explicit memory of the recent past.

In [ ]:
# ── Lag features ─────────────────────────────────────────────────────────────
features_df["lag_1"]  = features_df["daily_quantity_sold"].shift(1)   # yesterday
features_df["lag_7"]  = features_df["daily_quantity_sold"].shift(7)   # last week (same weekday)
features_df["lag_14"] = features_df["daily_quantity_sold"].shift(14)  # two weeks ago
features_df["lag_30"] = features_df["daily_quantity_sold"].shift(30)  # ~one month ago

print("Lag features added:")
print(features_df[["daily_quantity_sold",
                    "lag_1", "lag_7", "lag_14", "lag_30"]].head(35))


In [ ]:
# ── Correlation of lags with the target ──────────────────────────────────────
lag_cols = ["lag_1", "lag_7", "lag_14", "lag_30"]
correlations = features_df[["daily_quantity_sold"] + lag_cols].corr()["daily_quantity_sold"].drop("daily_quantity_sold")

plt.figure(figsize=(7, 4))
correlations.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Correlation of Lag Features with Daily Sales")
plt.ylabel("Pearson Correlation")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nCorrelation values:")
print(correlations)


## 3. Encoding Categorical Columns

Models need **numbers**, not strings. We first extract useful calendar features (day-of-week, month, etc.) and then encode any remaining categorical columns.

### 3a. Extract calendar features from the date index

In [ ]:
# ── Calendar features from the datetime index ────────────────────────────────
features_df["day_of_week"]  = features_df.index.dayofweek      # 0=Mon … 6=Sun
features_df["day_of_month"] = features_df.index.day
features_df["week_of_year"] = features_df.index.isocalendar().week.astype(int)
features_df["month"]        = features_df.index.month
features_df["quarter"]      = features_df.index.quarter
features_df["is_weekend"]   = (features_df["day_of_week"] >= 5).astype(int)

print("Calendar features:")
print(features_df[["day_of_week", "day_of_month",
                    "week_of_year", "month",
                    "quarter", "is_weekend"]].head(10))


### 3b. Label Encoding

Use **Label Encoding** when a column has an **ordinal** (ordered) relationship — e.g., `low < medium < high`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Example: create a synthetic 'season' column to demonstrate label encoding
season_map = {1: "Winter", 2: "Winter", 3: "Spring",
              4: "Spring",  5: "Spring", 6: "Summer",
              7: "Summer",  8: "Summer", 9: "Autumn",
              10: "Autumn", 11: "Autumn", 12: "Winter"}

features_df["season"] = features_df["month"].map(season_map)
print("Season column (before encoding):")
print(features_df["season"].value_counts())

le = LabelEncoder()
features_df["season_encoded"] = le.fit_transform(features_df["season"])

print("\nLabel encoding mapping:")
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} → {code}")

print("\nSample rows:")
print(features_df[["season", "season_encoded"]].head(10))


### 3c. One-Hot Encoding

Use **One-Hot Encoding** for **nominal** (unordered) categories — e.g., day-of-week names. Each category becomes its own binary column.

In [ ]:
# ── One-hot encode day_of_week ────────────────────────────────────────────────
day_names = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
features_df["day_name"] = features_df["day_of_week"].map(day_names)

ohe_days = pd.get_dummies(features_df["day_name"], prefix="day", dtype=int)
features_df = pd.concat([features_df, ohe_days], axis=1)

print("One-hot encoded day columns:")
print(features_df[[c for c in features_df.columns if c.startswith("day_")]].head(10))


### 3d. Cyclic Encoding

Calendar values like **month (1–12)** or **day-of-week (0–6)** are *cyclic* — December is as close to January as February is.
Representing them with sine/cosine captures this circular structure.

In [ ]:
# ── Cyclic encoding for month and day_of_week ─────────────────────────────────
features_df["month_sin"] = np.sin(2 * np.pi * features_df["month"] / 12)
features_df["month_cos"] = np.cos(2 * np.pi * features_df["month"] / 12)

features_df["dow_sin"]   = np.sin(2 * np.pi * features_df["day_of_week"] / 7)
features_df["dow_cos"]   = np.cos(2 * np.pi * features_df["day_of_week"] / 7)

# Visualise the cyclic encoding for month
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc = axes[0].scatter(features_df["month_cos"], features_df["month_sin"],
                     c=features_df["month"], cmap="hsv", s=10)
plt.colorbar(sc, ax=axes[0], label="Month")
axes[0].set_title("Cyclic Encoding – Month")
axes[0].set_xlabel("cos"); axes[0].set_ylabel("sin")

sc2 = axes[1].scatter(features_df["dow_cos"], features_df["dow_sin"],
                      c=features_df["day_of_week"], cmap="tab10", s=10)
plt.colorbar(sc2, ax=axes[1], label="Day of Week")
axes[1].set_title("Cyclic Encoding – Day of Week")
axes[1].set_xlabel("cos"); axes[1].set_ylabel("sin")
plt.tight_layout()
plt.show()

print("Cyclic feature sample:")
print(features_df[["month", "month_sin", "month_cos",
                   "day_of_week", "dow_sin", "dow_cos"]].head(8))


## 4. Drop Rows with NaN (created by lags/rolling windows)

The first N rows will contain `NaN` values because there is no history to look back on. We drop them before splitting.

In [ ]:
print(f"Shape before dropping NaNs: {features_df.shape}")
print(f"Total NaN cells: {features_df.isna().sum().sum()}")

features_df.dropna(inplace=True)

print(f"Shape after  dropping NaNs: {features_df.shape}")
print(f"Total NaN cells after drop: {features_df.isna().sum().sum()}")


## 5. Train / Test Split

For time-series data we **never shuffle** — we split chronologically.
The model trains on the past and is evaluated on the future.

> Rule of thumb: **80 % train / 20 % test** (or use a fixed cut-off date).

In [ ]:
# ── Chronological 80/20 split ─────────────────────────────────────────────────
target_col  = "daily_quantity_sold"
feature_cols = [c for c in features_df.columns
                if c not in [target_col, "season", "day_name"]]   # drop raw categoricals

X = features_df[feature_cols]
y = features_df[target_col]

split_idx = int(len(features_df) * 0.80)
split_date = features_df.index[split_idx]

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Split date      : {split_date.date()}")
print(f"Training period : {X_train.index.min().date()}  →  {X_train.index.max().date()}  ({len(X_train)} days)")
print(f"Test period     : {X_test.index.min().date()}   →  {X_test.index.max().date()}   ({len(X_test)} days)")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")


In [ ]:
# ── Visualise the train / test split ─────────────────────────────────────────
plt.figure(figsize=(14, 5))
plt.plot(y_train.index, y_train, label="Train", color="steelblue")
plt.plot(y_test.index,  y_test,  label="Test",  color="orange")
plt.axvline(split_date, color="red", linestyle="--", label=f"Split ({split_date.date()})")
plt.title("Train / Test Split – Daily Quantity Sold")
plt.xlabel("Date")
plt.ylabel("Quantity Sold")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Final Feature Matrix Summary

In [ ]:
print("=== WEEK 2 FEATURE SUMMARY ===\n")
print(f"Total features  : {len(feature_cols)}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples    : {len(X_test)}")
print("\nFeature columns:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

print("\nFirst 5 rows of final training feature matrix:")
print(X_train.head())


# Week 3 – Model Training & Evaluation

We train and compare four models on the feature matrix built in Week 2:
1. **Baseline** – Naïve last-week carry-forward (benchmark)
2. **Linear Regression** – simple parametric baseline
3. **Random Forest** – ensemble of decision trees
4. **XGBoost** – gradient-boosted trees (often best on tabular time-series)

All models are evaluated on the **held-out test set** using MAE, RMSE, and R².

In [ ]:
# ── Install XGBoost if not already present ───────────────────────────────
try:
    import xgboost
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print("All libraries ready.")

## 1. Baseline Model – Naïve Lag-7 Forecast

The simplest possible forecast: predict today's sales = sales **7 days ago**.
Any real model must beat this to be worth using.

In [ ]:
# ── Naïve baseline: predict = lag_7 ─────────────────────────────────────
y_pred_baseline = X_test["lag_7"]

mae_base  = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = mean_squared_error(y_test, y_pred_baseline) ** 0.5
r2_base   = r2_score(y_test, y_pred_baseline)

print("=== Naïve Baseline (Lag-7) ===")
print(f"  MAE  : {mae_base:.2f}")
print(f"  RMSE : {rmse_base:.2f}")
print(f"  R²   : {r2_base:.4f}")

## 2. Linear Regression

In [ ]:
# ── Linear Regression ────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr) ** 0.5
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Linear Regression ===")
print(f"  MAE  : {mae_lr:.2f}")
print(f"  RMSE : {rmse_lr:.2f}")
print(f"  R²   : {r2_lr:.4f}")

## 3. Random Forest

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = mean_squared_error(y_test, y_pred_rf) ** 0.5
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== Random Forest ===")
print(f"  MAE  : {mae_rf:.2f}")
print(f"  RMSE : {rmse_rf:.2f}")
print(f"  R²   : {r2_rf:.4f}")

## 4. XGBoost

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
y_pred_xgb = xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = mean_squared_error(y_test, y_pred_xgb) ** 0.5
r2_xgb   = r2_score(y_test, y_pred_xgb)

print("=== XGBoost ===")
print(f"  MAE  : {mae_xgb:.2f}")
print(f"  RMSE : {rmse_xgb:.2f}")
print(f"  R²   : {r2_xgb:.4f}")

## 5. Model Comparison

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
import pandas as pd

results = pd.DataFrame({
    "Model" : ["Naïve Baseline", "Linear Regression", "Random Forest", "XGBoost"],
    "MAE"   : [mae_base,  mae_lr,  mae_rf,  mae_xgb],
    "RMSE"  : [rmse_base, rmse_lr, rmse_rf, rmse_xgb],
    "R²"    : [r2_base,   r2_lr,   r2_rf,   r2_xgb],
})
results = results.sort_values("RMSE").reset_index(drop=True)
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ── Bar chart comparison ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#aec6cf", "#77b5b5", "#2196F3", "#0d47a1"]
models = results["Model"].tolist()

for ax, metric in zip(axes, ["MAE", "RMSE", "R²"]):
    vals = results[metric].tolist()
    bars = ax.bar(models, vals, color=colors, edgecolor="white")
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticklabels(models, rotation=20, ha="right", fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Model Comparison on Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Prediction vs Actual (Best Model)

In [ ]:
# ── Find best model by RMSE and plot predictions ────────────────────────
best_name = results.iloc[0]["Model"]
pred_map  = {
    "Naïve Baseline"   : y_pred_baseline,
    "Linear Regression": y_pred_lr,
    "Random Forest"    : y_pred_rf,
    "XGBoost"          : y_pred_xgb,
}
y_pred_best = pred_map[best_name]

plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values,      label="Actual",
         color="steelblue", linewidth=1.5)
plt.plot(y_test.index, y_pred_best,        label=f"Predicted ({best_name})",
         color="darkorange", linewidth=1.5, linestyle="--")
plt.title(f"Actual vs Predicted – {best_name} (Test Period)",
          fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Daily Quantity Sold")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest & XGBoost)

In [ ]:
# ── Feature importance from both tree models ─────────────────────────────
import numpy as np

TOP_N = 15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name in zip(axes,
                            [rf,  xgb],
                            ["Random Forest", "XGBoost"]):
    importances = model.feature_importances_
    idx  = np.argsort(importances)[::-1][:TOP_N]
    cols = [X_train.columns[i] for i in idx]
    vals = importances[idx]

    ax.barh(cols[::-1], vals[::-1], color="steelblue", edgecolor="white")
    ax.set_title(f"Top {TOP_N} Features – {name}", fontweight="bold")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 8. Residual Analysis (Best Model)

In [ ]:
# ── Residuals over time and distribution ─────────────────────────────────
residuals = y_test.values - np.array(y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Residuals over time
axes[0].plot(y_test.index, residuals, color="steelblue", linewidth=1)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_title(f"Residuals Over Time – {best_name}", fontweight="bold")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual (Actual − Predicted)")

# Residual distribution
axes[1].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Residual Distribution", fontweight="bold")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print(f"Residual mean  : {residuals.mean():.4f}  (close to 0 = unbiased)")
print(f"Residual std   : {residuals.std():.4f}")
print(f"Max overshoot  : {residuals.max():.2f}")
print(f"Max undershoot : {residuals.min():.2f}")